<a id="index"></a>
# 📑 Python Interview Notes — Index

All interview-ready topics in this notebook, quick-jump style:

- **[Topic 1 — Shallow vs Deep Copy](#topic-1)** — assignment vs `copy.copy()` vs `copy.deepcopy()`
- **[Topic 2 — `is` vs `==` and `id()`](#topic-2)** — identity vs equality, proving shared references
- **[Topic 3 — Pass by Reference and Value](#topic-3)** — how Python actually passes function arguments
- **[Topic 4 — Rebinding vs Mutating](#topic-4)** — the general rule behind every reference gotcha in Python
- **[Topic 5 — Type Annotation in Python](#topic-5)** — type hints, `typing` module, and why Python never actually enforces them

---

<a id="topic-1"></a>
# 🟦 TOPIC 1 — Shallow vs Deep Copy

[⬆ Back to Index](#index)

## 🎯 Learning Goal

By the end of this note, I will understand the difference between assignment, shallow copy, and deep copy in Python, why "copying" a list doesn't always give you an independent copy, and how to use the `copy` module to avoid nasty bugs caused by shared references.

---

## 🤔 What is it?

When you copy something in Python, there are actually three very different outcomes:

- **Assignment (`=`)** doesn't copy anything at all — it just gives the same object a second name.
- **Shallow copy** creates a new outer object, but the things *inside* it (like nested lists) are still shared with the original.
- **Deep copy** creates a completely independent copy — the outer object AND everything inside it are duplicated.

> 🧸 Think of a list like a box holding smaller boxes inside it. Assignment gives you a second label for the same box. A shallow copy gives you a new outer box, but the smaller boxes inside are still the *same* smaller boxes — open one from either box and you'll find the same stuff. A deep copy gives you a brand new outer box AND brand new smaller boxes inside, so nothing is shared anymore.

---

## ❓ Why do we need it?

If you assume `list2 = list1` gives you two separate lists, you'll be surprised when changing `list2` also changes `list1` — because they're really the same object underneath. And even `copy.copy()` (shallow copy) can trick you if your list contains other lists, dictionaries, or objects inside it. Understanding shallow vs deep copy stops you from accidentally corrupting data you didn't mean to touch.

---

## 🧠 Key Idea

- `=` never copies — it just points a new variable at the same object.
- Shallow copy (`copy.copy()`, `.copy()`, `list()`, slicing `[:]`) duplicates only the outer container.
- Deep copy (`copy.deepcopy()`) duplicates the outer container AND every nested object inside it, recursively.
- The difference only matters when your data has "objects inside objects" — like a list of lists.
- If your data is flat (just numbers or strings, no nesting), shallow copy behaves exactly like deep copy.

---

## 📚 Important Terms

| Term | Simple Meaning | Example |
|------|----------------|----------|
| Reference | A variable pointing to the same object in memory | `b = a` |
| Shallow Copy | New outer object, shared inner objects | `copy.copy(a)` |
| Deep Copy | New outer object AND new inner objects | `copy.deepcopy(a)` |
| `copy` module | Python's built-in tool for making shallow/deep copies | `import copy` |
| Mutable | Can be changed after creation (lists, dicts) | `[1, 2, 3]` |
| Immutable | Cannot be changed after creation (numbers, strings, tuples) | `"hello"`, `42` |

---

## 🔄 How it Works

```mermaid
flowchart TD

A[Original List] -->|"list2 = list1"| B["Same object, two names<br/>(assignment)"]
A -->|"copy.copy(list1)"| C["New outer list<br/>but nested items still shared"]
A -->|"copy.deepcopy(list1)"| D["New outer list<br/>AND new nested items"]
```

- **Assignment** — no new box is created at all, just a second label on the same box.
- **Shallow copy** — a new outer box is created, but it's filled with the *same* inner boxes as the original.
- **Deep copy** — a new outer box is created, and Python also goes inside and makes fresh copies of every inner box.

---

## 🌍 Real-Life Example

Imagine a **filing cabinet (list)** with folders (nested lists) inside it.

- **Assignment** is like giving someone the exact same cabinet key — there's still only one cabinet.
- **Shallow copy** is like buying a new cabinet, but instead of new folders, you just put the *same physical folders* from the old cabinet into it. Edit a folder from either cabinet, and both "see" the change, because it's the same folder.
- **Deep copy** is like buying a new cabinet AND photocopying every folder to put inside it. Now nothing is shared — editing one cabinet's folder never affects the other.

---

## 💻 Technical Example

### 1️⃣ Assignment — no copy at all

Both `list1` and `list2` point to the exact same list in memory. Changing one changes the other.

In [ ]:
list1 = [1, 2, [3, 4]]
list2 = list1  # just another name for the SAME list

list2.append(100)
print("list1:", list1)  # also changed!
print("list2:", list2)
print("Same object?", list1 is list2)

### 2️⃣ Shallow copy — new outer list, shared inner list

`copy.copy()` (or `.copy()`, or `list1[:]`) makes `list1` and `list2` different objects at the top level. But the nested list `[3, 4]` inside is still the SAME object in both — so mutating it from one side shows up on the other.

In [ ]:
import copy

list1 = [1, 2, [3, 4]]
list2 = copy.copy(list1)  # shallow copy

print("Same outer list?", list1 is list2)          # False - different outer list
print("Same inner list?", list1[2] is list2[2])     # True  - inner list is shared!

# Adding a top-level item only affects list2 - this is fine
list2.append(100)
print("\nAfter appending 100 to list2:")
print("list1:", list1)
print("list2:", list2)

# But mutating the NESTED list affects BOTH - this is the trap!
list2[2].append(999)
print("\nAfter appending 999 to list2's nested list:")
print("list1:", list1)  # list1's nested list changed too!
print("list2:", list2)

### 3️⃣ Deep copy — fully independent copy

`copy.deepcopy()` copies the outer list AND recursively copies everything nested inside it. Now nothing is shared, and mutating one never affects the other.

In [ ]:
import copy

list1 = [1, 2, [3, 4]]
list2 = copy.deepcopy(list1)  # deep copy

print("Same outer list?", list1 is list2)          # False
print("Same inner list?", list1[2] is list2[2])     # False - inner list is ALSO a new copy!

# Now mutating the nested list in list2 does NOT affect list1
list2[2].append(999)
print("\nAfter appending 999 to list2's nested list:")
print("list1:", list1)  # unchanged
print("list2:", list2)

---

## 🖼 Visual Representation

```
list1 = [1, 2, [3, 4]]

Assignment (list2 = list1):
list1 ──┐
        ├──► [1, 2, [3, 4]]
list2 ──┘

Shallow Copy (copy.copy(list1)):
list1 ──► [1, 2, ●] ──┐
                       ├──► [3, 4]   (SAME inner list)
list2 ──► [1, 2, ●] ──┘

Deep Copy (copy.deepcopy(list1)):
list1 ──► [1, 2, ●] ──► [3, 4]        (original inner list)
list2 ──► [1, 2, ●] ──► [3, 4]        (brand new inner list, same values)
```

---

## ⚖ Comparison

| | Assignment (`=`) | Shallow Copy | Deep Copy |
|---|---|---|---|
| New outer object? | ❌ No | ✅ Yes | ✅ Yes |
| New nested objects? | ❌ No | ❌ No (shared) | ✅ Yes |
| Speed | Instant | Fast | Slower (more work) |
| Safe for flat lists (no nesting)? | ❌ No | ✅ Yes | ✅ Yes |
| Safe for nested lists/dicts? | ❌ No | ❌ No | ✅ Yes |
| How to do it | `b = a` | `copy.copy(a)`, `a.copy()`, `a[:]` | `copy.deepcopy(a)` |

---

## 💡 Easy Trick to Remember

> 📌 **Shallow = skin deep, Deep = all the way down.**
>
> A shallow copy only copies the "skin" (the outer container). A deep copy copies the skin AND every organ inside it, no matter how deeply nested.

---

## ⚠ Common Misconceptions

❌ `list2 = list1` creates a copy of the list.
✅ It only creates a second name for the same list — no copying happens at all.

❌ `copy.copy()` always gives you a fully independent list.
✅ It only copies the outer list — nested lists/dicts inside are still shared with the original.

❌ Shallow copy is "wrong" and you should always use deep copy.
✅ Shallow copy is perfectly fine (and faster) when your data has no nested mutable objects inside it.

❌ Slicing (`list1[:]`) and `list()` always behave differently from `copy.copy()`.
✅ For lists, `list1[:]`, `list(list1)`, `list1.copy()`, and `copy.copy(list1)` all do the same thing — a shallow copy.

---

## 🔍 Interview Questions

- What's the difference between `list2 = list1` and `list2 = list1.copy()`?
- What is a shallow copy, and when can it cause unexpected bugs?
- What is a deep copy, and when should you use it over a shallow copy?
- Does shallow copy vs deep copy matter for a list of numbers like `[1, 2, 3]`? Why or why not?
- How would you deep copy a dictionary that contains lists as values?

---

## 📝 Quick Revision

- `=` doesn't copy anything — it just creates a second label for the same object.
- Shallow copy creates a new outer container, but nested objects inside are still shared.
- Deep copy creates a completely independent copy, all the way down through nested objects.
- Shallow copy tools for lists: `copy.copy()`, `.copy()`, `list()`, slicing `[:]`.
- Deep copy tool: `copy.deepcopy()` from the `copy` module.
- The difference only matters when data is nested (list of lists, dict of lists, etc.).
- For flat data (no nesting), shallow and deep copy behave identically.
- Mutating a shared nested object through one variable silently affects the other — this is the classic shallow copy bug.

---

## 🎓 Cheat Sheet

| Concept | One-Line Meaning |
|----------|------------------|
| Assignment (`=`) | Same object, two names — no copy |
| Shallow Copy | New outer object, shared nested objects |
| Deep Copy | New outer object, new nested objects (fully independent) |
| `copy.copy(x)` | Shallow copy of `x` |
| `copy.deepcopy(x)` | Deep copy of `x` |
| `x[:]` / `x.copy()` / `list(x)` | Shallow copy shortcuts for lists |

---

## 📖 Related Topics

Since this topic is **Shallow vs Deep Copy**, next recommended topics:

- [Lists](09_List.ipynb)
- [Dictionary](11_Dictionary.ipynb)
- Mutable vs Immutable Data Types
- [OOP Part 1](13_Oops_part1.ipynb) (objects and references)
- Function Arguments (pass by reference vs pass by value in Python)

---

## ✅ Key Takeaways

1. `list2 = list1` never copies anything — both names point to the exact same object in memory.
2. A shallow copy (`copy.copy()`, `.copy()`, `[:]`) creates a new outer container, but nested objects inside are still shared with the original.
3. A deep copy (`copy.deepcopy()`) creates a fully independent copy, including every nested object.
4. Shallow vs deep copy only matters when your data contains mutable objects nested inside other objects, like a list of lists.
5. Mutating a shared nested list through a "copy" is the classic bug — always ask: does my data have anything nested inside it?

---

<a id="topic-2"></a>
# 🟦 TOPIC 2 — `is` vs `==` and the `id()` Function

[⬆ Back to Index](#index)

## 🤔 What is it?

- `==` checks if two things have the **same value** (content).
- `is` checks if two things are **the exact same object** in memory (same identity).
- `id()` gives you a number — the object's actual memory address — so you can prove whether two variables point to the same object or just look alike.

> 🧸 Think of two identical twins wearing the same outfit. `==` asks "do they look the same?" (yes). `is` asks "are they literally the same person?" (no — two different bodies). `id()` is like checking each twin's fingerprint — different fingerprints mean different people, even if they're dressed identically.

This connects directly to shallow vs deep copy: `is` and `id()` are exactly how you can *prove*, in code, whether a "copy" is really a new object or just another name for the same one.

---

## ❓ Why do we need it?

If you only use `==`, you can't tell whether two variables are truly independent objects or secretly the same object. This matters a lot with mutable objects like lists — two lists can be `==` (same content) but NOT `is` (different objects), or they can be both `==` and `is` (literally the same object, as with plain assignment).

---

## 📚 Important Terms

| Term | Simple Meaning | Example |
|------|----------------|----------|
| `==` | Compares values/content | `[1,2] == [1,2]` → `True` |
| `is` | Compares identity (same object in memory) | `a is b` |
| `id(obj)` | Returns the unique memory address of an object | `id(a)` |
| Identity | "Are they literally the same object?" | Checked with `is` or by comparing `id()` |
| Equality | "Do they have the same value?" | Checked with `==` |

---

## 💻 Technical Example

In [ ]:
# == checks VALUE, is checks IDENTITY

a = [1, 2, 3]
b = [1, 2, 3]   # looks the same, but a fresh separate list
c = a           # same object as a (assignment)

print("a == b :", a == b)   # True  -> same content
print("a is b :", a is b)   # False -> different objects in memory
print()
print("a == c :", a == c)   # True  -> same content
print("a is c :", a is c)   # True  -> literally the same object

print()
print("id(a):", id(a))
print("id(b):", id(b))
print("id(c):", id(c))

## 🔎 Use Case: Proving Shallow Copy with `id()`

This is the real power move — instead of just *believing* a shallow copy shares nested objects, use `id()` on each index to actually prove it. Compare the outer list's `id`, then loop through every index and compare the `id` of each element between the original and the copy.

In [ ]:
import copy

original = [10, 20, [30, 40]]
shallow  = copy.copy(original)
deep     = copy.deepcopy(original)

print("Outer list id -> original:", id(original), " shallow:", id(shallow), " deep:", id(deep))
print("(all different outer objects, since all 3 are separate top-level lists)\n")

# Walk through EVERY index and compare id + data side by side
print(f"{'Index':<6}{'Data (orig)':<15}{'id(orig)':<20}{'id(shallow)':<20}{'id(deep)':<20}")
for index in range(len(original)):
    orig_item = original[index]
    shallow_item = shallow[index]
    deep_item = deep[index]
    print(f"{index:<6}{str(orig_item):<15}{id(orig_item):<20}{id(shallow_item):<20}{id(deep_item):<20}")

print("\nNotice index 2 (the nested list [30, 40]):")
print("shallow[2] is original[2] :", shallow[2] is original[2])  # True -> shared!
print("deep[2] is original[2]    :", deep[2] is original[2])     # False -> independent

## 🌍 Real-Life Example

Think of `id()` like a **student roll number** and `==` like comparing **marks**. Two different students (different roll numbers, i.e. different `id()`) can score the exact same marks (`==` is `True`). But they're still two different students (`is` is `False`). Only if you're looking at the same roll number twice (same `id()`) are you actually looking at the same student (`is` is `True`).

## 💡 Easy Trick to Remember

> 📌 **`==` = "Do you look the same?" | `is` = "Are you the same?" | `id()` = the proof.**
>
> If two variables are truly the same object, `is` returns `True` and `id()` gives back the exact same number for both.

## ⚠ Common Misconceptions

❌ `is` and `==` always give the same result.
✅ They usually match for numbers/strings due to Python's internal caching, but for lists, dicts, and custom objects they can easily disagree — always use `==` for value checks and `is` for identity checks (like `x is None`).

❌ `id()` is just a random number with no real meaning.
✅ `id()` reflects the object's actual location in memory (in CPython) — it's how Python itself decides whether two variables reference the same object.

❌ Comparing `original[2]` and `shallow[2]` with `==` proves they're shared.
✅ `==` only proves the *values* match. To prove they're the *same object* (truly shared, not just equal), you need `is` or matching `id()`.

## 🔍 Interview Questions

- What's the difference between `is` and `==` in Python?
- Why does `a = 5; b = 5; a is b` often return `True`, but `a = [5]; b = [5]; a is b` returns `False`?
- How would you use `id()` to prove whether a copy is shallow or deep?
- Why is `x is None` the recommended way to check for `None`, instead of `x == None`?

---

<a id="topic-3"></a>
# 🟦 TOPIC 3 — Pass by Reference and Value

[⬆ Back to Index](#index)

## 🎯 Learning Goal

By the end of this note, I will understand how Python actually passes arguments into functions (it's neither pure "pass by value" nor pure "pass by reference"), why changing an immutable argument inside a function doesn't affect the caller, why mutating a list/dict argument DOES affect the caller, and how to use `id()` to prove exactly what's happening.

---

## 🤔 What is it?

Python doesn't use "pass by value" (like C, where a full copy is made) or "pass by reference" (like C++ references, where the variable itself is aliased). Instead, Python uses **"pass by object reference"** (also called **pass by assignment**):

- When you call a function, the parameter becomes a new name pointing to the **same object** the argument points to.
- What happens next depends on whether that object is **mutable** or **immutable**, and whether the function **rebinds** the name or **mutates** the object.

> 🧸 Think of it like handing someone a **sticky note with an address written on it**, not the actual house. If they scribble a new address on their own copy of the sticky note (rebinding), your original sticky note still points to your house — nothing changes for you. But if they walk to that address and repaint the house (mutating), the house itself changed — and since it's the same house, you'll see the new paint too when you go back.

---

## ❓ Why do we need it?

If you assume Python behaves like Java (pass by value for primitives) or like C++ (pass by reference with `&`), you'll get surprised in both directions: expecting a list to stay unchanged after passing it to a function (it might not), or expecting an integer to change after passing it to a function (it never will). Understanding this model prevents confusing, hard-to-track bugs — especially with lists and dictionaries passed into functions.

---

## 🧠 Key Idea

- Python always passes the **reference to the object**, never a value copy and never a true C++-style alias.
- **Immutable objects** (`int`, `float`, `str`, `tuple`, `bool`, `frozenset`) — can't be changed in place, so any "change" inside the function is really a **rebind**, which only affects the local name, not the caller's variable.
- **Mutable objects** (`list`, `dict`, `set`, custom objects) — CAN be changed in place. If the function **mutates** them (`.append()`, `[i] = x`), the caller sees the change. If the function **rebinds** them (`x = [...]`), the caller does NOT see the change.
- The golden rule: **mutation is visible outside the function, rebinding is not** — no matter whether the object is mutable or immutable.

---

## 📚 Important Terms

| Term | Simple Meaning | Example |
|------|----------------|----------|
| Pass by Value | Caller gets a full independent copy (Python does NOT do this) | C's `int` arguments |
| Pass by Reference | Caller's variable itself is aliased (Python does NOT do this) | C++'s `int&` arguments |
| Pass by Object Reference | Parameter points to the same object as the argument (what Python actually does) | `def f(x):` shares `x` with the caller's object |
| Mutable | Can be changed in place, keeping the same `id()` | `list`, `dict`, `set` |
| Immutable | Cannot be changed in place — any "change" creates a new object | `int`, `str`, `tuple` |
| Mutate | Change the contents of an object without creating a new one | `my_list.append(4)` |
| Rebind | Point a name at a completely new object | `my_list = [1, 2, 3]` |

---

## 🔄 How it Works

```mermaid
flowchart TD

A["Call function: f(x)"] --> B["Parameter inside f points<br/>to the SAME object as x"]
B --> C{"What does f do?"}
C -->|"Mutates it<br/>(.append, [i]=val)"| D["Object changes in place<br/>Caller sees the change"]
C -->|"Rebinds it<br/>(x = new_value)"| E["Parameter now points to a<br/>NEW object — caller's variable<br/>still points to the old one"]
```

- Calling `f(x)` never copies `x` — the parameter starts out pointing to the exact same object.
- If the function **mutates** the object (changes it without creating a new one), the caller's variable sees it too, because it's still the same object.
- If the function **rebinds** the parameter (assigns it a brand-new object), that only affects the local name — the caller's variable is untouched.

---

## 🌍 Real-Life Example

Imagine you lend a friend your **shopping list** (a mutable list).

- If your friend **crosses off an item and adds a new one** on the same piece of paper (mutation), and hands it back — you'll see those changes, because it was the same paper the whole time.
- If your friend instead **throws your list away and writes a brand new one** on a fresh piece of paper (rebinding), your original list at home is completely unaffected — you never had that new paper.

Now imagine you hand your friend a **single number written on a card**, like your age.

- Numbers can't be "edited in place" in Python — your friend can only cross it out and write a new number (which is really a rebind). Either way, your original card is never touched, because they were only ever holding a copy of the *reference* to your number, and numbers can't mutate.

---

## 💻 Technical Example

### 1️⃣ Immutable argument (`int`) — the caller is never affected

Numbers can't be mutated, so `num += 1` inside the function is really `num = num + 1` — a rebind that creates a brand new integer object local to the function.

In [ ]:
def try_to_change(num):
    print("  inside (before): id =", id(num), " value =", num)
    num += 1  # this REBINDS num to a new int object, doesn't mutate the old one
    print("  inside (after):  id =", id(num), " value =", num)

age = 25
print("outside (before): id =", id(age), " value =", age)
try_to_change(age)
print("outside (after):  id =", id(age), " value =", age)  # unchanged!

### 2️⃣ Mutable argument (`list`), mutated — the caller IS affected

`.append()` changes the list **in place** — the `id()` stays the same the whole time. Since the caller's variable points to that same object, it sees the change too.

In [ ]:
def add_item(shopping_list):
    print("  inside (before): id =", id(shopping_list), " value =", shopping_list)
    shopping_list.append("milk")  # MUTATES the same object, no rebind
    print("  inside (after):  id =", id(shopping_list), " value =", shopping_list)

cart = ["bread", "eggs"]
print("outside (before): id =", id(cart), " value =", cart)
add_item(cart)
print("outside (after):  id =", id(cart), " value =", cart)  # changed!

### 3️⃣ Mutable argument (`list`), rebound — the caller is NOT affected

This is the trap. Even though lists are mutable, writing `shopping_list = [...]` doesn't mutate anything — it points the *local* parameter name at a brand new list. The caller's variable never finds out.

In [ ]:
def replace_list(shopping_list):
    print("  inside (before): id =", id(shopping_list), " value =", shopping_list)
    shopping_list = ["only", "this"]  # REBINDS the local name to a NEW list
    print("  inside (after):  id =", id(shopping_list), " value =", shopping_list)

cart = ["bread", "eggs"]
print("outside (before): id =", id(cart), " value =", cart)
replace_list(cart)
print("outside (after):  id =", id(cart), " value =", cart)  # unchanged!

---

## 🖼 Visual Representation

```
Case 1 - Immutable, "changed" inside function:
age (outside) ──► 25
num (inside, before)  ──► 25          (same object, id matches)
num (inside, after)   ──► 26          (NEW object - rebind, id changed)
age (outside, after)  ──► 25          (untouched)

Case 2 - Mutable, mutated inside function:
cart (outside, before) ──► ["bread","eggs"]
shopping_list (inside) ──► ["bread","eggs"]     (same object, same id)
        .append("milk") mutates THIS object
cart (outside, after)  ──► ["bread","eggs","milk"]   (sees the change - same id throughout)

Case 3 - Mutable, rebound inside function:
cart (outside, before) ──► ["bread","eggs"]
shopping_list (inside, before) ──► ["bread","eggs"]      (same object, same id)
shopping_list (inside, after)  ──► ["only","this"]       (NEW object - rebind, id changed)
cart (outside, after)  ──► ["bread","eggs"]              (untouched)
```

---

## ⚖ Comparison

| | Immutable (`int`, `str`, `tuple`) | Mutable, Mutated (`.append()`, `[i]=x`) | Mutable, Rebound (`x = [...]`) |
|---|---|---|---|
| Same object throughout? | ❌ No (any "change" rebinds) | ✅ Yes | ❌ No |
| `id()` before vs after (inside) | Different | Same | Different |
| Caller sees the change? | ❌ Never | ✅ Yes | ❌ No |
| Example | `num += 1` | `my_list.append(4)` | `my_list = [1, 2, 3]` |

---

## 💡 Easy Trick to Remember

> 📌 **"Mutate travels back, rebind stays local."**
>
> If you change the *contents* of an object (mutate), the caller feels it. If you point the name at something new (rebind), only the function's own local copy of the name changes — the caller never knows.

---

## ⚠ Common Misconceptions

❌ Python is "pass by reference," like C++.
✅ Python passes a reference to the object, but rebinding that reference inside the function never affects the caller — true pass-by-reference (like C++'s `&`) would let the function change what the caller's variable points to. Python can't do that.

❌ Python is "pass by value," like C.
✅ No copy of the object is made when calling a function — mutating a mutable argument (like a list) inside the function absolutely does affect the caller, which pure pass-by-value would never allow.

❌ Lists are always changed by functions, and numbers never are.
✅ It depends on what the function actually does: mutating a list changes it for the caller, but *rebinding* a list parameter (`x = [...]`) does not — the mutable/immutable label alone doesn't decide the outcome.

❌ `x += 1` always mutates `x` in place.
✅ For immutable types like `int`, `+=` creates a brand new object and rebinds the name — it never mutates the original number (numbers can't be mutated at all).

---

## 🔍 Interview Questions

- Is Python "pass by value" or "pass by reference"? What's the correct term for what it actually does?
- Why does appending to a list inside a function affect the caller's list, but reassigning an integer inside a function does not?
- What's the difference, inside a function, between mutating a parameter and rebinding it?
- If you pass a list into a function and the function does `my_list = my_list + [4]`, does the caller's list change? Why or why not?
- How would you use `id()` to prove whether a function mutated or rebound its argument?

---

## 📝 Quick Revision

- Python uses **pass by object reference** — not pass by value, not pass by reference.
- The parameter inside a function starts out pointing to the exact same object as the caller's argument.
- **Mutating** an object (`.append()`, `[i] = x`) changes it in place — the caller sees the change, because it's still the same object (`id()` unchanged).
- **Rebinding** a name (`x = new_value`) points the *local* name at a new object — the caller's variable is untouched.
- Immutable objects (`int`, `str`, `tuple`) can only ever be "rebound," never mutated — so functions can never change them for the caller.
- Mutable objects (`list`, `dict`, `set`) can go either way — check whether the function mutates or rebinds to know the outcome.
- `id()` is the tool that proves which one happened — compare it before and after inside the function.

---

## 🎓 Cheat Sheet

| Concept | One-Line Meaning |
|----------|------------------|
| Pass by Object Reference | Python's actual model — parameter shares the same object as the argument |
| Mutate | Change contents in place — caller sees it, `id()` stays the same |
| Rebind | Point the name at a new object — caller doesn't see it, `id()` changes |
| Immutable types | `int`, `float`, `str`, `tuple`, `bool` — can only be rebound |
| Mutable types | `list`, `dict`, `set` — can be mutated OR rebound |
| Proof tool | `id()` — compare before/after to see what really happened |

---

## 📖 Related Topics

Since this topic is **Pass by Reference and Value**, next recommended topics:

- Shallow vs Deep Copy (same underlying reference concepts, above in this notebook)
- Mutable vs Immutable Data Types
- [Functions](08_Function.ipynb) (default arguments, `*args`, `**kwargs`)
- [Lists](09_List.ipynb) and [Dictionary](11_Dictionary.ipynb)
- `is` vs `==` and the `id()` function (above in this notebook)

---

## ✅ Key Takeaways

1. Python doesn't do pure "pass by value" or pure "pass by reference" — it does **pass by object reference**, where the parameter shares the caller's object.
2. **Mutating** an argument inside a function (`.append()`, editing an index) changes it for the caller too, since it's the same object.
3. **Rebinding** an argument inside a function (`x = new_value`) never affects the caller — it only repoints the local name.
4. Immutable types (`int`, `str`, `tuple`) can only be rebound, never mutated — so a function can never change them for the caller.
5. Use `id()` before and after inside a function to prove, in code, whether a mutation or a rebind actually happened.

---

<a id="topic-4"></a>
# 🟦 TOPIC 4 — Rebinding vs Mutating

[⬆ Back to Index](#index)

## 🎯 Learning Goal

By the end of this note, I will understand the general rule of "rebind vs mutate" that sits underneath shallow/deep copy AND pass-by-object-reference, why `x += y` doesn't always do the same thing as `x = x + y`, and two classic interview gotchas that trip people up: mutable default arguments, and rebinding a global variable inside a function.

---

## 🤔 What is it?

Every Python variable is just a **name** pointing at an **object** — never a box holding a value. Given that, there are only ever two things you can do to a name:

- **Rebind** — point the name at a *different* object. The old object is untouched; only the name moved.
- **Mutate** — reach into the object the name currently points to and change its *contents*, without creating a new object. The name still points at the same object, but that object looks different now.

> 🧸 Think of a name like a **luggage tag** on a suitcase. Rebinding is taking the tag off one suitcase and clipping it onto a different suitcase — the first suitcase still exists, untouched, just without your tag on it anymore. Mutating is keeping the tag on the SAME suitcase, but opening it up and swapping out what's inside.

This is the single idea that explains shallow/deep copy (Topic 1), `is`/`==` (Topic 2), and function argument behavior (Topic 3) — they're all just this one rule applied in different situations.

---

## ❓ Why do we need it?

Without this distinction, code like `x += [1]` looks harmless, but whether it mutates or rebinds depends entirely on the type of `x` — and that difference decides whether other variables pointing at the same object get silently affected. This single rule also explains two of the most common real interview gotchas in Python: the mutable default argument bug, and the `UnboundLocalError` you get when you forget the `global` keyword.

---

## 🧠 Key Idea

- Immutable objects (`int`, `float`, `str`, `tuple`, `bool`, `frozenset`) can **only ever be rebound** — they have no in-place mutation methods at all.
- Mutable objects (`list`, `dict`, `set`, custom objects) support **both** — plain `=` rebinds, but methods like `.append()`, `.update()`, `.add()`, or index assignment `x[i] = v` mutate.
- `x = x + y` **always rebinds** `x` to a brand new object — this works for every type.
- `x += y` mutates **if the type supports in-place addition** (like `list.__iadd__`, which calls `extend`), otherwise it silently falls back to rebinding (like `int`, `str`, `tuple`).
- Function default arguments are evaluated **once**, at `def` time, not on every call — if that default is a mutable object, every call that doesn't supply its own argument mutates the *same* shared object.
- Assigning to a variable anywhere inside a function makes Python treat it as **local** for that entire function — to rebind an outer/global variable from inside a function, you must explicitly declare `global` (or `nonlocal` for enclosing function scope).

---

## 📚 Important Terms

| Term | Simple Meaning | Example |
|------|----------------|----------|
| Rebind | Point a name at a new object | `x = [1, 2]` |
| Mutate | Change an object's contents in place | `x.append(3)` |
| `+=` (in-place add) | Mutates if the type supports it, else rebinds | `list += [1]` mutates, `int += 1` rebinds |
| Mutable default argument | A `def f(x=[])` default created once and reused across calls | Classic Python gotcha |
| `global` keyword | Tells Python "this name refers to the module-level variable" | `global counter` |
| `UnboundLocalError` | Error from using a local name before assigning it, because Python already decided it's local | Caused by rebinding a global without `global` |

---

## 🔄 How it Works

```mermaid
flowchart TD

A["x += y"] --> B{"Does type of x support<br/>in-place addition?"}
B -->|"Yes (list, dict.update, set.add)"| C["MUTATE — same id(),<br/>every other reference sees it"]
B -->|"No (int, str, tuple)"| D["REBIND — new id(),<br/>only x's own name is affected"]
```

- `x += y` first checks if `x`'s type defines an in-place operation (`__iadd__`).
- If yes (lists, and similar), it mutates the existing object — `id(x)` stays the same.
- If no (numbers, strings, tuples — anything immutable), Python quietly does `x = x + y` instead — a plain rebind, and `id(x)` changes.

---

## 🌍 Real-Life Example

Think of a **shared Google Doc** (mutable object, like a list) versus a **printed PDF** (immutable object, like a string or tuple).

- Editing the Google Doc **directly** (mutating) means everyone with the link sees your edits instantly — same document, new content.
- "Editing" a PDF really means **saving a brand new PDF** with your changes (rebinding) — anyone still holding a link to the original PDF sees no difference at all, because you never touched their copy.

---

## 💻 Technical Example

### 1️⃣ `x += y` vs `x = x + y` — the same-looking code that behaves differently

For a `list`, `+=` mutates in place (`id()` unchanged). For an `int`, `+=` is really a rebind (`id()` changes). `x = x + y` always rebinds, no matter the type.

In [ ]:
# LIST: += MUTATES (same id)
nums = [1, 2, 3]
print("list before +=  : id =", id(nums), " value =", nums)
nums += [4]  # mutates in place -> calls list.extend() under the hood
print("list after  +=  : id =", id(nums), " value =", nums)

print()

# LIST: x = x + y REBINDS (new id)
nums2 = [1, 2, 3]
print("list before x+y : id =", id(nums2), " value =", nums2)
nums2 = nums2 + [4]  # rebinds -> a brand new list is created
print("list after  x+y : id =", id(nums2), " value =", nums2)

print()

# INT: += always rebinds, because int has no in-place mutation
num = 10
print("int before +=   : id =", id(num), " value =", num)
num += 1  # int can't mutate -> this is really num = num + 1
print("int after  +=   : id =", id(num), " value =", num)

### 2️⃣ Why this matters — a second variable watching the same list

Because `+=` mutates a list in place, any other variable pointing at that same list sees the change too. `x = x + y` never has this side effect, because it rebinds `x` to a new object that `alias` never finds out about.

In [ ]:
original = [1, 2, 3]
alias = original  # both names point to the SAME list

original += [4]  # MUTATES -> alias sees it too
print("After += :")
print("original:", original)
print("alias:   ", alias)          # changed! still the same object
print("Same object?", original is alias)

print()

original2 = [1, 2, 3]
alias2 = original2

original2 = original2 + [4]  # REBINDS -> alias2 does NOT see it
print("After x = x + y :")
print("original2:", original2)
print("alias2:   ", alias2)        # unchanged! original2 now points elsewhere
print("Same object?", original2 is alias2)

### 3️⃣ The mutable default argument trap

`def add_item(item, cart=[])` looks like it gives every call a fresh empty list. It doesn't — the default list is created **once**, when the function is defined, and every call that skips the `cart` argument mutates that *same* shared list.

In [ ]:
# BUGGY version - mutable default argument
def add_item_buggy(item, cart=[]):
    cart.append(item)  # mutates the SAME default list every time
    return cart

print(add_item_buggy("apple"))   # expected: ["apple"]
print(add_item_buggy("banana"))  # expected: ["banana"], but the old cart leaked in!

print()

# FIXED version - use None as the default, create a fresh list inside
def add_item_fixed(item, cart=None):
    if cart is None:
        cart = []  # a brand new list every call - no sharing
    cart.append(item)
    return cart

print(add_item_fixed("apple"))
print(add_item_fixed("banana"))  # correctly isolated this time

### 4️⃣ Rebinding a global variable needs the `global` keyword

Mutating a global list works fine without any keyword, because mutation doesn't need to rebind the name. But **rebinding** a global name from inside a function requires declaring `global` first — otherwise Python treats that name as local for the whole function, and reading it before assignment throws `UnboundLocalError`.

In [ ]:
shared_list = [1, 2, 3]
counter = 0

def mutate_global():
    shared_list.append(4)  # MUTATION - works fine, no keyword needed

def rebind_global():
    global counter        # required - tells Python "use the module-level counter"
    counter += 1           # REBIND (int can't mutate) - needs `global` to affect the outer name

mutate_global()
print("shared_list after mutate_global():", shared_list)  # changed - mutation doesn't need `global`

rebind_global()
rebind_global()
print("counter after two rebind_global() calls:", counter)  # 2 - global let the rebind escape

def broken_rebind():
    # no `global` here -> Python treats `counter` as LOCAL for this whole function
    counter += 1  # this line will fail: local `counter` is read before it's assigned
    return counter

try:
    broken_rebind()
except UnboundLocalError as e:
    print("\nUnboundLocalError:", e)

---

## 🖼 Visual Representation

```
x += y :

  list    -> [MUTATE]  same id, same suitcase, new contents
  dict    -> [MUTATE]  (.update() works the same way)
  set     -> [MUTATE]  (.add() / |= works the same way)
  int     -> [REBIND]  new id, name moves to a new suitcase
  str     -> [REBIND]  new id
  tuple   -> [REBIND]  new id (tuples can't even += in place)

x = x + y :

  ANY type -> [REBIND]  always creates a brand new object, no exceptions
```

---

## ⚖ Comparison

| | `x = x + y` | `x += y` on a mutable (`list`) | `x += y` on an immutable (`int`, `str`) |
|---|---|---|---|
| Rebinds or mutates? | Always rebinds | Mutates | Rebinds (silently) |
| `id(x)` changes? | ✅ Yes | ❌ No | ✅ Yes |
| Other aliases see the change? | ❌ No | ✅ Yes | ❌ No |

---

## 💡 Easy Trick to Remember

> 📌 **"Rebind moves the tag, mutate opens the suitcase."**
>
> If you're not sure which one just happened, check `id()` before and after. Same `id()` → it mutated. Different `id()` → it rebound.

---

## ⚠ Common Misconceptions

❌ `x += y` and `x = x + y` always do the exact same thing.
✅ For mutable types like lists, `+=` mutates in place while `x = x + y` creates a brand new object — they behave identically for the variable `x` itself, but very differently for any other variable that was pointing at the same original object.

❌ `def f(x=[]):` gives every call to `f()` a fresh empty list.
✅ The default list is created once, at function definition time, and reused (and potentially mutated) across every call that doesn't pass its own argument — the standard fix is `def f(x=None): x = x or []`.

❌ You need `global` any time you touch a global variable inside a function.
✅ You only need `global` to **rebind** it (`counter = counter + 1`). Mutating it in place (`shared_list.append(4)`) works without `global`, because no new name-to-object binding is happening.

❌ `UnboundLocalError` means the variable was never created anywhere.
✅ It means Python decided the name is local to that function (because it's assigned somewhere inside it) — and you tried to read it before that local assignment ran, shadowing the outer variable of the same name.

---

## 🔍 Interview Questions

- What's the actual difference between rebinding a name and mutating an object in Python?
- Why can `x += y` behave differently from `x = x + y` for a list, but not for an integer?
- What is the mutable default argument bug, and how do you fix it?
- Why does forgetting the `global` keyword cause an `UnboundLocalError` instead of just using the outer variable's value?
- If a function mutates a list passed into it, does it need `global` or `nonlocal`? Why or why not?

---

## 📝 Quick Revision

- Every Python variable is a name pointing at an object — rebinding moves the name, mutating changes the object.
- Immutable types (`int`, `str`, `tuple`) can only ever be rebound.
- Mutable types (`list`, `dict`, `set`) can be either mutated (`.append()`, `.update()`) or rebound (`x = [...]`).
- `x += y` mutates for types with in-place operators (like `list`), but silently rebinds for everything else.
- `x = x + y` always rebinds, regardless of type.
- A mutable default function argument is created once and shared across every call that omits it — use `None` and create the object inside the function instead.
- Mutating a global variable inside a function needs no keyword; rebinding one requires `global` (or `nonlocal` inside nested functions).

---

## 🎓 Cheat Sheet

| Concept | One-Line Meaning |
|----------|------------------|
| Rebind | Point a name at a new object — old object untouched |
| Mutate | Change an object's contents — same object, new insides |
| `x += y` on mutable | Mutates in place, `id()` unchanged |
| `x += y` on immutable | Rebinds, `id()` changes |
| `x = x + y` | Always rebinds, for any type |
| Mutable default arg fix | `def f(x=None): x = x or []` |
| `global` | Needed only when rebinding (not mutating) a module-level name inside a function |

---

## 📖 Related Topics

Since this topic is **Rebinding vs Mutating**, next recommended topics:

- Shallow vs Deep Copy (Topic 1, above in this notebook)
- `is` vs `==` and `id()` (Topic 2, above in this notebook)
- Pass by Reference and Value (Topic 3, above in this notebook)
- Closures and Late Binding (why loop variables in closures capture the final value, not each iteration's value)
- `nonlocal` keyword and nested function scope

---

## ✅ Key Takeaways

1. Rebinding points a name at a new object; mutating changes the object a name already points to — every reference gotcha in Python boils down to this one distinction.
2. `x += y` mutates for types with in-place support (lists, dicts, sets) but silently rebinds for immutable types (ints, strings, tuples).
3. `x = x + y` always rebinds, no matter the type — this is the safe, predictable option when you don't want to affect other aliases.
4. A mutable default function argument (`def f(x=[])`) is created once and shared across calls — always default to `None` and build the mutable object inside the function body instead.
5. Mutating a global variable inside a function needs no special keyword, but rebinding one does — forgetting `global` causes an `UnboundLocalError`, not a silent read of the outer value.

---

<a id="topic-5"></a>
# 🟦 TOPIC 5 — Type Annotation in Python

[⬆ Back to Index](#index)

## 🎯 Learning Goal

By the end of this note, I will understand what type annotations (type hints) are, how to write them for variables and functions, what the `typing` module gives you (`Optional`, `Union`, `List`, `Dict`, etc.), and — most importantly — why Python never actually enforces any of it at runtime.

---

## 🤔 What is it?

A **type annotation** (also called a **type hint**) is a label you attach to a variable, function parameter, or return value saying what type it's *expected* to be. Python remains a **dynamically typed** language even with annotations — they're just metadata, a form of documentation that tools can read.

```python
age: int = 25

def greet(name: str) -> str:
    return "Hello, " + name
```

> 🧸 Think of type hints like the label on a moving box that says "FRAGILE — GLASSWARE." The label doesn't physically stop you from stuffing a brick in there instead. It's just a note for whoever's handling the box (you, your teammates, or a tool like a mover/checker) to catch mistakes *before* something breaks.

---

## ❓ Why do we need it?

Python doesn't require type hints to run — code works identically with or without them. They exist to solve human and tooling problems: making function signatures self-documenting, letting your editor autocomplete and warn you about mismatches, and letting external static type checkers (like `mypy`) catch bugs *before* you ever run the code — especially valuable in large codebases where you can't just "remember" every function's expected types.

---

## 🧠 Key Idea

- Type hints are **optional** and **not enforced** by the Python interpreter itself — Python will happily run code that violates its own hints.
- They come from **PEP 484** and are written using a colon for variables/parameters (`name: str`) and an arrow for return values (`-> str`).
- The `typing` module provides building blocks for more complex hints: `Optional[X]` (X or `None`), `Union[X, Y]` (X or Y), `List[X]`, `Dict[K, V]`, `Tuple[X, Y]`, and more.
- Enforcement only happens if you run a **separate static type checker** (like `mypy` or `pyright`) — that's a tool you run alongside your code, not something Python does automatically.
- All annotations are stored in a special `__annotations__` dictionary that you can inspect at runtime.

---

## 📚 Important Terms

| Term | Simple Meaning | Example |
|------|----------------|----------|
| Type Hint / Annotation | A label describing the expected type | `age: int` |
| Dynamically Typed | Python decides types at runtime, not before | `x = 5` then `x = "five"` both work |
| `typing` module | Standard library module with tools for complex hints | `from typing import Optional` |
| `Optional[X]` | "X, or `None`" | `Optional[str]` |
| `Union[X, Y]` | "X or Y" | `Union[int, str]` |
| Static Type Checker | An external tool that reads hints and flags mismatches *before* running | `mypy`, `pyright` |
| `__annotations__` | A dict where Python stores all the hints it saw | `func.__annotations__` |

---

## 🔄 How it Works

```mermaid
flowchart TD

A["Code with type hints"] --> B["Python interpreter"]
A --> C["Static type checker<br/>(mypy, pyright) — optional, separate step"]
B --> D["Runs normally —<br/>hints are IGNORED at runtime"]
C --> E["Reads hints, flags mismatches<br/>BEFORE the code ever runs"]
```

- The Python interpreter itself only ever looks at hints to store them in `__annotations__` — it never checks whether you actually followed them.
- A static type checker is a completely separate tool you choose to run — it's the only thing that turns hints into real errors, and only at check time, not runtime.

---

## 🌍 Real-Life Example

Think of a **recipe card** that says "2 cups flour" (a type hint). If you actually pour in 2 cups of sugar instead, the recipe card doesn't stop your hand — nothing physically prevents the mistake while you're cooking (that's Python at runtime). But if a friend **reviews your recipe card before you cook** and says "hey, that's supposed to be flour, not sugar" — that's the static type checker, catching the mismatch before it becomes a bad dish.

---

## 💻 Technical Example

### 1️⃣ Basic variable and function annotations

Variables get a `name: type` label. Functions annotate each parameter and the return type with `-> type`.

In [ ]:
# Variable annotations
age: int = 25
name: str = "Pratham"
price: float = 99.99
is_active: bool = True

# Function annotation: (a: int, b: int) -> int
def add(a: int, b: int) -> int:
    return a + b

print(add(2, 3))
print("age:", age, " name:", name, " price:", price, " is_active:", is_active)

### 2️⃣ Proof that Python never enforces type hints at runtime

`add()` is annotated to take two `int`s and return an `int`. Nothing stops you from calling it with strings — Python runs it anyway and just does whatever the `+` operator does for that type.

In [ ]:
def add(a: int, b: int) -> int:
    return a + b

# Hint says int, but we pass strings anyway - Python does NOT stop us
result = add("2", "3")
print("add('2', '3') =", result, " type =", type(result).__name__)
# "23" (string concatenation) - clearly not an int, and Python never complained

# Only a separate static type checker like mypy would flag this as an error
# BEFORE you even ran the code - Python itself just executes it as written

### 3️⃣ Inspecting hints with `__annotations__`

Every annotation you write is stored in a dictionary Python builds automatically — you can read it back at runtime, which is exactly how tools like `mypy` and IDEs know what you meant.

In [ ]:
def multiply(a: int, b: int) -> int:
    return a * b

print("Function annotations:", multiply.__annotations__)
# {'a': <class 'int'>, 'b': <class 'int'>, 'return': <class 'int'>}

age: int = 25
name: str = "Pratham"
print("Module-level annotations:", __annotations__)
# shows every variable annotation written at this level, e.g. {'age': <class 'int'>, 'name': <class 'str'>, ...}

### 4️⃣ The `typing` module — hints for more complex shapes

Plain types like `int` and `str` aren't enough for real code — you often need "this or `None`", "this or that", or "a list of these." The `typing` module (and, in modern Python, built-in generics like `list[int]`) covers that.

In [ ]:
from typing import Optional, Union, List, Dict, Tuple

# Optional[X] means "X, or None"
def find_user(user_id: int) -> Optional[str]:
    users = {1: "Pratham", 2: "Rohan"}
    return users.get(user_id)  # returns None if not found - the hint says that's allowed

print(find_user(1))   # "Pratham"
print(find_user(99))  # None - matches Optional[str]

# Union[X, Y] means "X or Y"
def stringify(value: Union[int, str]) -> str:
    return str(value)

print(stringify(42))
print(stringify("hello"))

# Generic containers: List[X], Dict[K, V], Tuple[X, Y]
def average(nums: List[int]) -> float:
    return sum(nums) / len(nums)

def get_scores() -> Dict[str, int]:
    return {"math": 90, "science": 85}

def get_point() -> Tuple[int, int]:
    return (3, 4)

print(average([10, 20, 30]))
print(get_scores())
print(get_point())

---

## 🖼 Visual Representation

```
def add(a: int, b: int) -> int:
                ^        ^      ^
                |        |      |
          param hint  param hint  return hint

At runtime -> Python: "noted, ignoring, running the function as-is"
At check time (mypy run) -> "add('2', '3') called with str, expected int - ERROR"
```

---

## ⚖ Comparison

| | Without Type Hints | With Type Hints (no checker run) | With Type Hints + `mypy`/`pyright` |
|---|---|---|---|
| Code runs the same? | ✅ Yes | ✅ Yes | ✅ Yes |
| Readable signatures? | ❌ Guess from usage | ✅ Yes | ✅ Yes |
| IDE autocomplete help? | ❌ Limited | ✅ Better | ✅ Better |
| Catches type bugs before running? | ❌ No | ❌ No | ✅ Yes |

---

## 💡 Easy Trick to Remember

> 📌 **"Hints are sticky notes, not security guards."**
>
> Python reads the note, files it away in `__annotations__`, and moves on. Only a checker you run separately (like `mypy`) actually acts like a guard and stops you.

---

## ⚠ Common Misconceptions

❌ Type hints make Python statically typed.
✅ Python stays dynamically typed no matter how many hints you add — hints are just metadata that tools can optionally read and enforce.

❌ Adding a wrong type hint (or violating one) will raise a `TypeError` at runtime.
✅ Python never validates hints at runtime — you'll only see an error if the underlying operation itself fails (like trying `"a" + 5`), which happens with or without hints.

❌ You need to install anything to use type hints.
✅ Basic hints (`int`, `str`, `List`, etc. via `typing`) work with plain Python — no extra install needed to *write* hints. You only need to install something (like `mypy`) if you want them *enforced* by a checker.

❌ `Optional[str]` means "this parameter is optional" (has a default value).
✅ `Optional[str]` only means "this value can be a `str` or `None`" — it says nothing about whether the parameter has a default. You still need `= None` separately if you want an actual default.

---

## 🔍 Interview Questions

- What is a type hint in Python, and does it change how the code runs?
- What's the difference between Python being "dynamically typed" and having type hints?
- What does `Optional[str]` mean, and how is it different from just giving a parameter a default value of `None`?
- If a function has a return type hint of `-> int` but actually returns a string, what happens when you run it?
- How would a tool like `mypy` use type hints that Python itself ignores?

---

## 📝 Quick Revision

- A type hint (annotation) labels the expected type of a variable, parameter, or return value.
- Syntax: `name: type` for variables/parameters, `-> type` for function return values.
- Python's interpreter stores hints in `__annotations__` but never checks or enforces them at runtime.
- The `typing` module adds `Optional[X]`, `Union[X, Y]`, `List[X]`, `Dict[K, V]`, `Tuple[X, Y]`, and more for complex shapes.
- Enforcement only comes from a separate static type checker (`mypy`, `pyright`) that you run as its own step.
- Type hints exist for humans and tools (readability, IDE autocomplete, catching bugs before running) — not for the Python runtime itself.

---

## 🎓 Cheat Sheet

| Concept | One-Line Meaning |
|----------|------------------|
| `x: int` | Variable annotation |
| `def f(a: int) -> int:` | Parameter and return type annotation |
| `Optional[X]` | X or `None` |
| `Union[X, Y]` | X or Y |
| `List[X]` / `Dict[K, V]` / `Tuple[X, Y]` | Generic container hints |
| `__annotations__` | Where Python stores all the hints it saw |
| `mypy` / `pyright` | Separate tools that actually enforce hints |

---

## 📖 Related Topics

Since this topic is **Type Annotation in Python**, next recommended topics:

- Rebinding vs Mutating (Topic 4, above in this notebook)
- Duck Typing and Dynamic Typing in Python
- Dataclasses (which lean heavily on type annotations)
- `mypy` basics and running a static type checker
- [Functions](08_Function.ipynb) (default arguments, `*args`, `**kwargs`)

---

## ✅ Key Takeaways

1. Type hints are optional metadata that describe expected types — Python's interpreter never checks or enforces them.
2. Syntax is `name: type` for variables/parameters and `-> type` for function return values.
3. The `typing` module covers complex shapes: `Optional[X]`, `Union[X, Y]`, `List[X]`, `Dict[K, V]`, `Tuple[X, Y]`.
4. Every hint you write ends up in `__annotations__`, which is how external tools (and you, via introspection) can read them back.
5. Real enforcement only happens if you run a separate static type checker like `mypy` or `pyright` — Python itself will run mismatched code without complaint.